# Grade 10 · Unit 1 — Cleaning a daily series

> **SYNTHETIC** air-quality table (25 blanks + four `-999` codes). Real data: `python tools/fetch_data.py kathmandu-air-quality`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

raw = pd.read_csv(data_file("kathmandu-air-quality", "air_daily_synthetic.csv"))
raw.info()

## Predict → Run: what is wrong with this table?

In [ ]:
print((raw["pm25"] == -999).sum(), "error codes;", raw["pm25"].isna().sum(), "blanks")
raw["pm25"].describe()

## Clean and log

In [ ]:
air = raw.copy()
air["date"] = pd.to_datetime(air["date"])
n_bad = int((air["pm25"] == -999).sum())
air["pm25"] = air["pm25"].replace(-999, np.nan)
log = [f"replaced {n_bad} sensor codes (-999) with NaN",
       f"{int(air['pm25'].isna().sum())} PM2.5 values are now missing (of {len(air)})"]
assert n_bad == 4 and air["pm25"].min() >= 0
print("\n".join(log))

## Summaries by time
**Predict:** which months will be most polluted?

In [ ]:
air["month"] = air["date"].dt.month
monthly = air.groupby("month")["pm25"].mean().round(1)
print(monthly)
monthly.plot(kind="bar", title="Mean PM2.5 by month (SYNTHETIC)", ylabel="µg/m³")
plt.show()

## Smooth with a rolling mean

In [ ]:
air = air.sort_values("date").set_index("date")
air["pm25_7d"] = air["pm25"].rolling(7, min_periods=4).mean()
air[["pm25", "pm25_7d"]].plot(figsize=(9, 3), alpha=0.8)
plt.show()

**Modify:** try a 30-day window. **Make:** compare weekdays vs weekends with `groupby(air.index.dayofweek)`. Write the cleaning log in your own words.